# TP4 – Chargement du lakehouse Iceberg (Bronze / Silver / Gold)

Ce notebook remplit l'entrepôt local (parcours B) à partir du jeu **TPCx-BB** fourni :

| Couche | Catalogue Iceberg | Contenu |
| --- | --- | --- |
| Bronze | `iceberg.bronze.*` | Les 23 tables du kit, lot initial + lot refresh, typées, sans filtre |
| Silver | `iceberg.silver.*` | `store_sales`, `store_returns`, `item`, `store`, `date_dim` nettoyés (règles UC-06), lot initial |
| Gold | `iceberg.gold.*` | Agrégats pour les analyses UC-08 |

Stockage : MinIO (`s3://warehouse/`), format de table Apache Iceberg (fichiers Parquet), catalogue REST, requêtable par Spark et Trino.

- Les fichiers source ne sont pas modifiés (montage en lecture seule).
- Les noms de colonnes et types observés viennent de l'inventaire du Jour 1 (`J1_Retail_Intelligence/01_UC01_besoin_exploration/preuves/colonnes.csv`), lui-même aligné sur la spécification TPCx-BB 1.7.0. Le nom n'est appliqué que si la largeur du fichier correspond.
- `customer/customer.csv` est une copie octet pour octet de `customer.dat` : seul le `.dat` est chargé. `item_marketprices_audit.csv` est un rapport du générateur, pas une table.
- Le notebook est rejouable : chaque table est recréée (`createOrReplace`).

Exécuter toutes les cellules dans l'ordre, dans le conteneur `lake` (`docker compose up -d`, Jupyter `http://localhost:8888/?token=tp`).

In [1]:
import csv
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TP4-chargement-lakehouse")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

SOURCES = {"initial": Path("/data/retail_project/data"), "refresh": Path("/data/retail_project/data_refresh")}
INVENTAIRE_COLONNES = Path("/opt/tp/J1_Retail_Intelligence/01_UC01_besoin_exploration/preuves/colonnes.csv")
LECTURE = {"sep": "|", "header": False, "quote": "\u0000", "escape": "\u0000", "mode": "PERMISSIVE"}

print("Python", sys.version.split()[0])
print("Spark", spark.version)
print("catalogue", spark.conf.get("spark.sql.catalog.iceberg.uri"), spark.conf.get("spark.sql.catalog.iceberg.warehouse"))

26/11/19 14:07:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Python 3.12.16
Spark 4.1.4


catalogue http://iceberg-rest:8181 s3://warehouse/


## Schémas

Correspondance des types observés au Jour 1 vers les types Iceberg :

| Type observé | Type Iceberg | Remarque |
| --- | --- | --- |
| entier | `bigint` | sauf codes postaux (`*_zip`) gardés en texte pour ne pas perdre les zéros de tête |
| decimal | `decimal(15,2)` | montants à 2 décimales |
| date | `date` | |
| vide | `date` si le nom contient `date`, sinon `string` | colonnes entièrement vides (`*_rec_end_date`) |
| texte, mixte, heure | `string` | un nombre isolé dans un libellé ne change pas le type |

Le cast est fait avec `try_cast` : une valeur non vide qui ne se convertit pas devient `null` **et est comptée** (contrôle `echecs_cast`).

In [2]:
def type_iceberg(nom, observe):
    observe = observe.split(" (")[0]
    if nom.endswith("_zip"):
        return "string"
    if observe == "entier":
        return "bigint"
    if observe == "decimal":
        return "decimal(15,2)"
    if observe == "date" or (observe == "vide" and "date" in nom):
        return "date"
    return "string"

schemas = {}
with open(INVENTAIRE_COLONNES, encoding="utf-8-sig") as f:
    for ligne in csv.DictReader(f):
        if ligne["lot"] != "initial":
            continue
        table = ligne["fichier"].removesuffix(".dat")
        schemas.setdefault(table, []).append((int(ligne["index"]), ligne["nom"], type_iceberg(ligne["nom"], ligne["type_observe"])))
schemas = {t: [(n, ty) for _, n, ty in sorted(cols)] for t, cols in sorted(schemas.items())}

print(len(schemas), "tables")
for t, cols in schemas.items():
    print(f"{t:24s} {len(cols):3d} colonnes")

23 tables
customer                  18 colonnes
customer_address          13 colonnes
customer_demographics      9 colonnes
date_dim                  28 colonnes
household_demographics     5 colonnes
income_band                3 colonnes
inventory                  4 colonnes
item                      22 colonnes
item_marketprices          6 colonnes
product_reviews            8 colonnes
promotion                 19 colonnes
reason                     3 colonnes
ship_mode                  6 colonnes
store                     29 colonnes
store_returns             20 colonnes
store_sales               23 colonnes
time_dim                  10 colonnes
warehouse                 14 colonnes
web_clickstreams           6 colonnes
web_page                  14 colonnes
web_returns               24 colonnes
web_sales                 34 colonnes
web_site                  26 colonnes


## Bronze – les 23 tables, lots initial et refresh

Une table Iceberg par table TPCx-BB. Colonnes techniques ajoutées : `_lot` (initial / refresh), `_source_fichier`, `_charge_le`. Aucune ligne n'est filtrée.

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bronze")

def lire_dat(chemin, colonnes):
    brut = spark.read.options(**LECTURE).csv("file://" + str(chemin))
    if len(brut.columns) != len(colonnes):
        raise ValueError(f"{chemin} : {len(brut.columns)} champs, {len(colonnes)} attendus")
    brut = brut.toDF(*[n for n, _ in colonnes])
    typees = [F.expr(f"try_cast(`{n}` as {ty})").alias(n) for n, ty in colonnes]
    echecs = sum(
        (F.when(F.col(n).isNotNull() & (F.trim(F.col(n)) != "") & F.expr(f"try_cast(`{n}` as {ty})").isNull(), 1).otherwise(0)
         for n, ty in colonnes if ty != "string"),
        F.lit(0),
    )
    return brut.select(*typees, echecs.alias("_echecs_cast"))

bilan_bronze = []
for table, colonnes in schemas.items():
    morceaux = []
    for lot, racine in SOURCES.items():
        fichier = racine / table / f"{table}.dat"
        if fichier.exists():
            morceaux.append(
                lire_dat(fichier, colonnes)
                .withColumn("_lot", F.lit(lot))
                .withColumn("_source_fichier", F.lit(str(fichier)))
            )
    df = morceaux[0]
    for m in morceaux[1:]:
        df = df.unionByName(m)
    df = df.withColumn("_charge_le", F.current_timestamp())
    df.drop("_echecs_cast").writeTo(f"iceberg.bronze.{table}").using("iceberg").createOrReplace()

    stats = df.groupBy("_lot").agg(F.count("*").alias("n"), F.sum("_echecs_cast").alias("e")).collect()
    par_lot = {r["_lot"]: r for r in stats}
    bilan_bronze.append((
        table,
        len(colonnes),
        par_lot["initial"]["n"] if "initial" in par_lot else 0,
        par_lot["refresh"]["n"] if "refresh" in par_lot else 0,
        sum(r["e"] for r in stats),
    ))
    print(f"{table:24s} initial={bilan_bronze[-1][2]:>10,} refresh={bilan_bronze[-1][3]:>8,} echecs_cast={bilan_bronze[-1][4]}")

customer                 initial=    99,000 refresh=   1,000 echecs_cast=0


customer_address         initial=    49,500 refresh=     500 echecs_cast=0


customer_demographics    initial= 1,920,800 refresh=       0 echecs_cast=0


date_dim                 initial=   109,573 refresh=       0 echecs_cast=0


household_demographics   initial=     7,200 refresh=       0 echecs_cast=0


income_band              initial=        20 refresh=       0 echecs_cast=0


... (13 outputs left out)

web_page                 initial=        60 refresh=       0 echecs_cast=0


web_returns              initial=    38,487 refresh=     400 echecs_cast=0


web_sales                initial=   668,052 refresh=   6,879 echecs_cast=0


web_site                 initial=        30 refresh=       0 echecs_cast=0


In [ ]:
bilan = spark.createDataFrame(bilan_bronze, "table string, colonnes int, lignes_initial long, lignes_refresh long, echecs_cast long")
bilan.orderBy("table").show(30, truncate=False)

def lignes_fichier(chemin):
    with open(chemin, "rb") as f:
        return sum(1 for _ in f)

ecarts = [
    (t, n, lignes_fichier(SOURCES["initial"] / t / f"{t}.dat"))
    for t, _, n, _, _ in bilan_bronze
]
ecarts = [e for e in ecarts if e[1] != e[2]]
print("tables dont le volume diffère du fichier :", ecarts or "aucune")

+----------------------+--------+--------------+--------------+-----------+
|table                 |colonnes|lignes_initial|lignes_refresh|echecs_cast|
+----------------------+--------+--------------+--------------+-----------+
|customer              |18      |99000         |1000          |0          |
|customer_address      |13      |49500         |500           |0          |
|customer_demographics |9       |1920800       |0             |0          |
|date_dim              |28      |109573        |0             |0          |
|household_demographics|5       |7200          |0             |0          |
|income_band           |3       |20            |0             |0          |
|inventory             |4       |23255100      |234900        |0          |
|item                  |22      |17820         |180           |0          |
|item_marketprices     |6       |89100         |900           |0          |
|product_reviews       |8       |89991         |909           |0          |
|promotion  

tables dont le volume diffère du fichier : aucune


## Silver – données nettoyées (lot initial)

Règles reprises du notebook `uc06_silver_gold.ipynb` (Jour 3) et étendues aux tables nécessaires à UC-08 :

| Table | Règle | Action |
| --- | --- | --- |
| `store_sales` | date, article, magasin, ticket, quantité, montant net non nuls ; `ss_quantity > 0` | lignes écartées et comptées |
| `store_sales` | doublon exact | supprimé et compté |
| `store_sales` | `ss_net_profit < 0` | conservé (résultat de ligne, pas une erreur) |
| `store_returns` | date, article, ticket, quantité non nuls ; `sr_return_quantity > 0` ; doublon exact | écarté / supprimé et compté |
| `item` | `i_item_sk`, `i_item_id` non nuls ; `i_item_sk` unique | écarté / dédoublonné |
| `store`, `date_dim` | clé de substitution unique et non nulle | contrôle, arrêt si écart |

Le lot refresh reste en Bronze : il n'est pas fusionné dans Silver, comme au Jour 3.

In [5]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.silver")

def bronze(table):
    return spark.table(f"iceberg.bronze.{table}").filter(F.col("_lot") == "initial").drop("_charge_le")

def nettoyer(df, requis, positifs=(), dedoublonner_sur=None):
    masque = F.lit(False)
    for c in requis:
        masque = masque | F.col(c).isNull()
    for c in positifs:
        masque = masque | (F.col(c) <= 0)
    gardees = df.filter(~masque)
    n_entree, n_ecartees = df.count(), df.filter(masque).count()
    sortie = gardees.dropDuplicates(dedoublonner_sur) if dedoublonner_sur else gardees.dropDuplicates()
    n_sortie = sortie.count()
    return sortie, {"entree": n_entree, "ecartees": n_ecartees, "doublons": n_entree - n_ecartees - n_sortie, "sortie": n_sortie}

regles = {
    "store_sales": dict(requis=["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_ticket_number", "ss_quantity", "ss_net_paid"], positifs=["ss_quantity"]),
    "store_returns": dict(requis=["sr_returned_date_sk", "sr_item_sk", "sr_ticket_number", "sr_return_quantity"], positifs=["sr_return_quantity"]),
    "item": dict(requis=["i_item_sk", "i_item_id"], dedoublonner_sur=["i_item_sk"]),
    "store": dict(requis=["s_store_sk"], dedoublonner_sur=["s_store_sk"]),
    "date_dim": dict(requis=["d_date_sk", "d_date"], dedoublonner_sur=["d_date_sk"]),
}

bilan_silver = []
for table, regle in regles.items():
    df, stats = nettoyer(bronze(table), **regle)
    df.writeTo(f"iceberg.silver.{table}").using("iceberg").createOrReplace()
    bilan_silver.append((table, stats["entree"], stats["ecartees"], stats["doublons"], stats["sortie"]))

spark.createDataFrame(bilan_silver, "table string, lignes_bronze long, ecartees long, doublons long, lignes_silver long").show(truncate=False)

ss = spark.table("iceberg.silver.store_sales")
print("profits négatifs conservés :", ss.filter(F.col("ss_net_profit") < 0).count())

+-------------+-------------+--------+--------+-------------+
|table        |lignes_bronze|ecartees|doublons|lignes_silver|
+-------------+-------------+--------+--------+-------------+
|store_sales  |667579       |0       |0       |667579       |
|store_returns|37902        |0       |0       |37902        |
|item         |17820        |0       |0       |17820        |
|store        |12           |0       |0       |12           |
|date_dim     |109573       |0       |0       |109573       |
+-------------+-------------+--------+--------+-------------+



profits négatifs conservés : 495971


## Contrôles d'intégrité Silver

Les jointures utilisées en Gold doivent conserver toutes les lignes de vente (relations vérifiées au Jour 1).

In [6]:
ss = spark.table("iceberg.silver.store_sales")
it = spark.table("iceberg.silver.item")
st = spark.table("iceberg.silver.store")
dd = spark.table("iceberg.silver.date_dim")

controles = [
    ("ss_item_sk -> item", ss.join(it, ss.ss_item_sk == it.i_item_sk, "left_anti").count()),
    ("ss_store_sk -> store", ss.join(st, ss.ss_store_sk == st.s_store_sk, "left_anti").count()),
    ("ss_sold_date_sk -> date_dim", ss.join(dd, ss.ss_sold_date_sk == dd.d_date_sk, "left_anti").count()),
]
for nom, orphelins in controles:
    print(f"{nom:30s} orphelins = {orphelins}")
if any(o for _, o in controles):
    raise SystemExit("relations non vérifiées : Gold non écrit")

ss_item_sk -> item             orphelins = 0
ss_store_sk -> store           orphelins = 0
ss_sold_date_sk -> date_dim    orphelins = 0


## Gold – agrégats pour UC-08

| Table | Granularité | Mesures | Usage |
| --- | --- | --- | --- |
| `ventes_magasin_par_produit` | un article (`i_item_sk`) | lignes, quantité, CA net, CA TTC, profit net | analyse produits / catégories (même définition qu'au Jour 3) |
| `ventes_magasin_mensuelles` | un magasin × un mois | lignes, tickets distincts, quantité, CA net, profit net | évolution temporelle, comparaison des magasins |
| `retours_magasin_mensuels` | un magasin × un mois | lignes de retour, quantité retournée, montant retourné | volume de retours |

Montants dans l'unité du fichier (non précisée par le kit). Un ticket est identifié par `ss_ticket_number` ; une ligne de vente est un article d'un ticket.

In [7]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.gold")

enrichi = (
    ss.join(it.select("i_item_sk", "i_item_id", "i_product_name", "i_category", "i_class"), ss.ss_item_sk == it.i_item_sk)
      .join(st.select("s_store_sk", "s_store_id", "s_store_name", "s_state"), ss.ss_store_sk == st.s_store_sk)
      .join(dd.select("d_date_sk", "d_date"), ss.ss_sold_date_sk == dd.d_date_sk)
      .withColumn("mois", F.date_format("d_date", "yyyy-MM"))
)

par_produit = (
    enrichi.groupBy("i_item_sk", "i_item_id", "i_product_name", "i_category", "i_class")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.sum("ss_quantity").alias("quantite"),
        F.sum("ss_net_paid").alias("ca_net"),
        F.sum("ss_net_paid_inc_tax").alias("ca_net_ttc"),
        F.sum("ss_net_profit").alias("profit_net"),
    )
)
par_produit.writeTo("iceberg.gold.ventes_magasin_par_produit").using("iceberg").createOrReplace()

mensuel = (
    enrichi.groupBy("mois", "s_store_sk", "s_store_id", "s_store_name", "s_state")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.countDistinct("ss_ticket_number").alias("nb_tickets"),
        F.sum("ss_quantity").alias("quantite"),
        F.sum("ss_net_paid").alias("ca_net"),
        F.sum("ss_net_profit").alias("profit_net"),
    )
)
mensuel.writeTo("iceberg.gold.ventes_magasin_mensuelles").using("iceberg").createOrReplace()

sr = spark.table("iceberg.silver.store_returns")
retours = (
    sr.join(dd.select("d_date_sk", "d_date"), sr.sr_returned_date_sk == dd.d_date_sk)
      .withColumn("mois", F.date_format("d_date", "yyyy-MM"))
      .groupBy("mois", "sr_store_sk")
      .agg(
          F.count("*").alias("nb_lignes_retour"),
          F.sum("sr_return_quantity").alias("quantite_retournee"),
          F.sum("sr_return_amt").alias("montant_retourne"),
      )
)
retours.writeTo("iceberg.gold.retours_magasin_mensuels").using("iceberg").createOrReplace()

for t in ["ventes_magasin_par_produit", "ventes_magasin_mensuelles", "retours_magasin_mensuels"]:
    print(t, spark.table(f"iceberg.gold.{t}").count(), "lignes")

ventes_magasin_par_produit 17820 lignes
ventes_magasin_mensuelles 720 lignes
retours_magasin_mensuels 948 lignes


## Contrôles de cohérence Gold

In [8]:
gp = spark.table("iceberg.gold.ventes_magasin_par_produit")
gm = spark.table("iceberg.gold.ventes_magasin_mensuelles")

ca_silver = ss.agg(F.sum("ss_net_paid")).first()[0]
verifs = [
    ("lignes Gold produit = lignes Silver", gp.agg(F.sum("nb_lignes")).first()[0], ss.count()),
    ("lignes Gold mensuel = lignes Silver", gm.agg(F.sum("nb_lignes")).first()[0], ss.count()),
    ("CA Gold produit = CA Silver", gp.agg(F.sum("ca_net")).first()[0], ca_silver),
    ("CA Gold mensuel = CA Silver", gm.agg(F.sum("ca_net")).first()[0], ca_silver),
]
for nom, a, b in verifs:
    print(f"{nom:40s} {a} / {b} -> {'OK' if a == b else 'ECART'}")

lignes Gold produit = lignes Silver      667579 / 667579 -> OK
lignes Gold mensuel = lignes Silver      667579 / 667579 -> OK
CA Gold produit = CA Silver              1150529586.37 / 1150529586.37 -> OK
CA Gold mensuel = CA Silver              1150529586.37 / 1150529586.37 -> OK


## Inventaire final de l'entrepôt

Les mêmes tables sont visibles depuis Trino : `docker compose exec trino trino --execute "SHOW TABLES FROM iceberg.gold"`.

In [9]:
for ns in ["bronze", "silver", "gold"]:
    tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN iceberg.{ns}").collect()]
    print(f"iceberg.{ns} ({len(tables)}) :", ", ".join(sorted(tables)))

iceberg.bronze (23) : customer, customer_address, customer_demographics, date_dim, household_demographics, income_band, inventory, item, item_marketprices, product_reviews, promotion, reason, ship_mode, store, store_returns, store_sales, time_dim, warehouse, web_clickstreams, web_page, web_returns, web_sales, web_site
iceberg.silver (5) : date_dim, item, store, store_returns, store_sales
iceberg.gold (3) : retours_magasin_mensuels, ventes_magasin_mensuelles, ventes_magasin_par_produit
